<a href="https://colab.research.google.com/github/rachelv2/mini-project-spotify/blob/main/spotify-main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas openpyxl

In [3]:
import pandas as pd

In [4]:
df = pd.read_excel('charts_group_project.xlsb')
df.head()

,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name
0,44875,au,169,280161,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up
1,44875,be,182,59603,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up
2,44875,ie,184,46016,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up
3,44875,cz,198,43421,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up
4,44868,be,159,62915,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up


In [5]:
df.nunique()

date                515
country              77
position            336
streams          375371
track_id          21518
artists           13686
artist_genres      7576
duration          17581
explicit              2
name              19274
dtype: int64

In [6]:
df.duplicated().sum() #no duplicates

np.int64(0)

In [7]:
df.info() #no missing values(only in artist name but we don't need that)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 10 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   date           1048575 non-null  int64 
 1   country        1048575 non-null  object
 2   position       1048575 non-null  int64 
 3   streams        1048575 non-null  int64 
 4   track_id       1048575 non-null  object
 5   artists        1048575 non-null  object
 6   artist_genres  1048575 non-null  object
 7   duration       1048575 non-null  int64 
 8   explicit       1048575 non-null  bool  
 9   name           1048388 non-null  object
dtypes: bool(1), int64(4), object(5)
memory usage: 73.0+ MB


In [8]:
#created a new column with just the year
df['date'] = pd.to_datetime(df['date'], origin='1899-12-30', unit='D')
df['year'] = df['date'].dt.year
df.head()

,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name,year
0,2022-11-10,au,169,280161,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022
1,2022-11-10,be,182,59603,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022
2,2022-11-10,ie,184,46016,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022
3,2022-11-10,cz,198,43421,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022
4,2022-11-03,be,159,62915,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022


In [9]:
!pip install pycountry

In [10]:
#fixing country names
import pycountry

def get_country_name(code):
    try:
        return pycountry.countries.get(alpha_2=code.upper()).name
    except:
        return code  # keeps values like 'global' unchanged

df['country_full'] = df['country'].apply(get_country_name)
df.head()

,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name,year,country_full
0,2022-11-10,au,169,280161,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Australia
1,2022-11-10,be,182,59603,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Belgium
2,2022-11-10,ie,184,46016,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Ireland
3,2022-11-10,cz,198,43421,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Czechia
4,2022-11-03,be,159,62915,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Belgium


In [12]:
#convert the string in 'artist' to a list
import ast

def parse_artist_string(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

#remove question mark
def clean_artist_list(artist_list):
    return [name.replace('?', '').strip() for name in artist_list if name.replace('?', '').strip() != '']

df['artist'] = df['artists'].apply(parse_artist_string).apply(clean_artist_list)

df['artist'][626791] #example where I knew it had question marks

['Seif Magdy']

In [13]:
#convert the duration to min:sec (created a new column)
df['duration_min_sec'] = df['duration'].apply(lambda x: f"{x//60000}:{(x%60000)//1000:02}")
df.head()

,date,country,position,streams,track_id,artists,artist_genres,duration,explicit,name,year,country_full,artist,duration_min_sec
0,2022-11-10,au,169,280161,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Australia,[Avicii],4:07
1,2022-11-10,be,182,59603,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Belgium,[Avicii],4:07
2,2022-11-10,ie,184,46016,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Ireland,[Avicii],4:07
3,2022-11-10,cz,198,43421,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Czechia,[Avicii],4:07
4,2022-11-03,be,159,62915,0nrRP2bk19rLc0orkWPQk2,['Avicii'],"['pop rap', 'pop', 'pop dance', 'dance pop', '...",247426,False,Wake Me Up,2022,Belgium,[Avicii],4:07


In [14]:
# analyse the number of explicit musics
df_clean = df[df['explicit'] == False]

df_explicit = df[df['explicit'] == True]

df['explicit'].value_counts()

explicit
False    800295
True     248280
Name: count, dtype: int64